In [4]:
import os
import time
import requests
import numpy as np
import pandas as pd

HUBS = [
    ("Kafr_ElSheikh", 31.11, 30.94), ("Dakahlia", 31.04, 31.38),
    ("Sharqia", 30.72, 31.67), ("Beheira", 31.03, 30.47),
    ("Gharbia", 30.79, 31.00), ("Menoufia", 30.55, 31.01),
    ("Qalyubia", 30.46, 31.18), ("Salhia_Reclaimed", 30.65, 31.95),
    ("Fayoum", 29.31, 30.84), ("Beni_Suef", 29.07, 31.09),
    ("Minya", 28.09, 30.75), ("Assiut", 27.18, 31.18),
    ("Sohag", 26.56, 31.69), ("Qena_Luxor", 25.69, 32.64),
]

PARAMS = "T2M,T2M_MIN,T2M_MAX,RH2M,ALLSKY_SFC_SW_DWN,PRECTOTCORR,WS2M,GWETTOP,GWETROOT,GWETPROF"
RENAME_MAP = {
    "T2M": "POWER_Temp_Mean_C",
    "T2M_MIN": "POWER_Temp_Min_C",
    "T2M_MAX": "POWER_Temp_Max_C",
    "RH2M": "POWER_Humidity_Pct",
    "ALLSKY_SFC_SW_DWN": "POWER_Solar_Rad_MJ_m2",
    "PRECTOTCORR": "POWER_Precip_mm_day",
    "WS2M": "POWER_Wind_Speed_ms",
    "GWETTOP": "POWER_Surface_Soil_Wetness",
    "GWETROOT": "POWER_RootZone_Soil_Wetness",
    "GWETPROF": "POWER_Profile_Soil_Moisture",
}

date_bins = pd.date_range("2016-01-01", "2025-12-31", freq="10D")
expected_dates = pd.to_datetime(date_bins[:-1])

def fetch_power(lat, lon, retries=4):
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters={PARAMS}&community=AG&longitude={lon}&latitude={lat}"
        "&start=20160101&end=20251231&format=JSON"
    )
    for attempt in range(retries):
        try:
            r = requests.get(url, headers={"User-Agent": "FieldShiftEgypt/1.0"}, timeout=60)
            js = r.json()
            if r.status_code == 200 and "properties" in js:
                return js["properties"]["parameter"]
        except Exception:
            pass
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"فشل الاتصال بخادم NASA POWER للإحداثيات ({lat}, {lon})")

dfs = []
print("📥 [1/6] بدء سحب بيانات NASA POWER لـ 14 محافظة...", flush=True)

for name, lat, lon in HUBS:
    raw = fetch_power(lat, lon)
    df = pd.DataFrame(raw).reset_index().rename(columns={"index": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.replace(-999.0, np.nan).rename(columns=RENAME_MAP)

    df["Bin"] = pd.cut(df["Date"], bins=date_bins, right=False, labels=expected_dates)
    df_10d = df.groupby("Bin", observed=True).mean(numeric_only=True).reindex(expected_dates).reset_index()
    df_10d.rename(columns={"index": "Date"}, inplace=True)
    df_10d["Governorate"] = name
    dfs.append(df_10d)
    print(f"   ✓ {name}", flush=True)
    time.sleep(0.8)

out = pd.concat(dfs, ignore_index=True).sort_values(["Governorate", "Date"]).reset_index(drop=True)
num_cols = out.select_dtypes(include=[np.number]).columns
out[num_cols] = out.groupby("Governorate")[num_cols].transform(lambda g: g.interpolate("linear").ffill().bfill())
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

cols = ["Governorate", "Date"] + [c for c in out.columns if c not in ["Governorate", "Date"]]
out = out[cols]

output_path = r"d:\vscode\NASA\DATA\01_NASA_POWER_Climate.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
out.to_csv(output_path, index=False)
print(f"✅ اكتمل الملف الأول: {output_path} | الأبعاد: {out.shape}\n")

📥 [1/6] بدء سحب بيانات NASA POWER لـ 14 محافظة...


   ✓ Kafr_ElSheikh
   ✓ Dakahlia
   ✓ Sharqia
   ✓ Beheira
   ✓ Gharbia
   ✓ Menoufia
   ✓ Qalyubia
   ✓ Salhia_Reclaimed
   ✓ Fayoum
   ✓ Beni_Suef
   ✓ Minya
   ✓ Assiut
   ✓ Sohag
   ✓ Qena_Luxor
✅ اكتمل الملف الأول: d:\vscode\NASA\DATA\01_NASA_POWER_Climate.csv | الأبعاد: (5110, 12)

